In [166]:
import pandas as pd

In [167]:
import numpy as np

In [168]:
df=pd.read_csv(r"D:\DATA ANALYSIS\Meta Ads Campaign Performance Analysis\Outputs\csv files for power bi\powerbi_main.csv")

In [169]:
df

,campaign_id,campaign_name,adset,platform,objective,status,start_date,end_date,impressions,reach,...,CTR,CPC,CPA,CVR,CPL,ROI,ROAS,Frequency,Decision,duration_days
0,C027,Hyderabad Conversion,Interest - Technology,Facebook,Leads,COMPLETED,2026-02-18,2026-02-25,83426,68019,...,0.52,7.16,1557.16,0.46,311.43,-19.45,0.81,1.23,Discontinue,7
1,C055,Ahmedabad Sales,Lookalike 1%,Instagram,Leads,PAUSED,2026-04-18,2026-05-06,107945,72764,...,0.68,7.55,691.76,1.09,307.45,-79.09,0.21,1.48,Discontinue,18
2,C050,Jaipur Lead Generation,Interest - Technology,Instagram,Conversions,ACTIVE,2026-01-14,2026-02-10,14694,13073,...,1.31,30.75,NaN,0.00,269.79,15.13,1.15,1.12,Pause,27
3,C093,Mumbai Sales,Interest - Technology,Facebook,Leads,ACTIVE,2026-04-25,2026-05-22,230437,198773,...,1.17,1.93,113.44,1.70,57.98,-10.40,0.90,1.16,Discontinue,27
4,C151,Mumbai Sales,Interest - Business,Instagram,Leads,PAUSED,2026-01-15,2026-02-12,239711,197146,...,0.87,1.63,141.01,1.16,27.97,16.91,1.17,1.22,Continue,28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
252,C202,Pune Education,Lookalike 1%,Instagram,Sales,PAUSED,2026-02-27,2026-03-07,228309,165885,...,0.27,5.00,510.45,0.98,65.16,-29.22,0.71,1.38,Discontinue,8
253,C169,Hyderabad Conversion,Engaged Users,Instagram,Traffic,PAUSED,2026-04-04,2026-05-05,159526,136594,...,0.88,1.42,77.16,1.84,14.33,-78.20,0.22,1.17,Discontinue,31
254,C186,Hyderabad Conversion,Broad Audience,Facebook,Traffic,ACTIVE,2026-06-22,2026-07-12,50000,42394,...,1.22,9.58,450.88,2.12,189.08,18.86,1.19,1.18,Continue,20
255,C036,Pune Education,Retargeting,Facebook,Conversions,COMPLETED,2026-05-31,2026-06-11,105814,59807,...,0.71,8.16,1536.27,0.53,149.88,-15.00,0.85,1.77,Discontinue,11


In [170]:
##BUDGET REALLOCATION
#Budget for next 6 months = ₹10 Lakh
#Top 10 Continue campaigns = 50% of whole budget

In [171]:
Wasted_spend = df[df['Decision']=='Discontinue']['spend'].sum()
print(f"Wasted spend (Discontinue): ₹{Wasted_spend:,.2f}")

Wasted spend (Discontinue): ₹609,345.47


In [172]:
budget_for_6_months=1000000
budget=budget_for_6_months*0.5
print(f"New Budget:{budget:,.2f}")

New Budget:500,000.00


In [173]:
#ROAS of Top 10

In [174]:
df_scale = df[df['Decision'] == 'Continue']
top10=df_scale.nlargest(10,'ROAS').copy()
top10_spend=top10['spend'].sum()
top10_revenue=top10['revenue'].sum()
current_roas  = top10_revenue / top10_spend

print(f"Top 10 spend : {top10_spend:,.2f}")
print(f"Top 10 Revenue : {top10_revenue:,.2f}")
print(f"Current ROAS : {current_roas:,.2f}x")

Top 10 spend : 32,184.88
Top 10 Revenue : 278,490.44
Current ROAS : 8.65x


In [175]:
#ROAS Decay

In [176]:
scale_factor=budget/top10_spend
print(f"Scaling factor : {scale_factor:,.2f}")

Scaling factor : 15.54


In [177]:
# Formula: new_roas = current_roas * (scale_factor ** -0.3)
adjusted_roas = current_roas *(scale_factor**-0.3)

print(f"Current ROAS : {current_roas:,.2f}")
print(f"Adjusted ROAS : {adjusted_roas:,.2f}")

Current ROAS : 8.65
Adjusted ROAS : 3.80


In [178]:
# estimate revenue & roi

In [179]:
estimate_revenue = budget * adjusted_roas

print(f"Estimate Revenue : ₹{estimate_revenue:,.2f}")

Estimate Revenue : ₹1,899,909.74


In [180]:
estimate_profit = estimate_revenue-budget
estimate_roi = (estimate_profit/budget)*100

print(f"Profit: ₹{estimate_profit:,.2f}")
print(f"ROI : {estimate_roi:,.2f}%")

Profit: ₹1,399,909.74
ROI : 279.98%


In [182]:
##Optimize Pause
#Pause Campaigns = 20% of whole budget
#34 "Pause" campaigns

In [183]:
pause_df = df[df['Decision'] == 'Pause'].copy()

print(f'Total Pause Campaigns: {len(pause_df)}')

Total Pause Campaigns: 34


In [184]:
#4 Quadrants

In [185]:
q1 = pause_df[(pause_df['CTR'] > 1.0) & (pause_df['CVR'] > 1.0)]
q2 = pause_df[(pause_df['CTR'] > 1.0) & (pause_df['CVR'] <= 1.0)]
q3 = pause_df[(pause_df['CTR'] <= 1.0) & (pause_df['CVR'] > 1.0)]
q4 = pause_df[(pause_df['CTR'] <= 1.0) & (pause_df['CVR'] <= 1.0)]

In [186]:
print(f"Q1: {len(q1):2}\nQ2: {len(q2):2}\nQ3: {len(q3):2}\nQ4: {len(q4):2}")
print(f"Sum: {len(q1) + len(q2) + len(q3) + len(q4)}")

Q1:  0
Q2: 16
Q3:  1
Q4: 17
Sum: 34


In [187]:
print("  Optimize Pause — ALLOCATION")

  Optimize Pause — ALLOCATION


In [188]:
print(f"\n📌 Sub-Bucket Breakdown:")
print(f"  Q1 Healthy:        {len(q1)} → promote to Bucket 1")
print(f"  Q2 Low CVR only:   {len(q2)} → funnel fix")
print(f"  Q3 Low CTR only:   {len(q3)} → creative fix")
print(f"  Q4 Broken both:    {len(q4)} → deprioritize")


📌 Sub-Bucket Breakdown:
  Q1 Healthy:        0 → promote to Bucket 1
  Q2 Low CVR only:   16 → funnel fix
  Q3 Low CTR only:   1 → creative fix
  Q4 Broken both:    17 → deprioritize


In [189]:
TOTAL_BUDGET = 200000

In [190]:
weights = {'creative':   len(q3) * 1.0,'audience':   len(q2) * 1.0,'funnel':     min(10, len(q2) + len(q4)) * 2.0,'reserve':    1.0}
total_weight = sum(weights.values())
allocations = {k: TOTAL_BUDGET * v / total_weight for k, v in weights.items()}

In [191]:
print(f"\n💰 Adaptive Allocation (₹2,00,000):")
for k, v in allocations.items():
    print(f"  {k.title():10}: ₹{v:>12,.2f} ({v/TOTAL_BUDGET*100:.0f}%)")


💰 Adaptive Allocation (₹2,00,000):
  Creative  : ₹    5,263.16 (3%)
  Audience  : ₹   84,210.53 (42%)
  Funnel    : ₹  105,263.16 (53%)
  Reserve   : ₹    5,263.16 (3%)


In [192]:
#Project Revenue

In [193]:
creative = allocations['creative']
audience = allocations['audience']
funnel = allocations['funnel']
reserve = allocations['reserve']

In [194]:
creative_roas = 2.5
audience_roas = 3.0
funnel_roas = 3.5
reserve_roas = 4.0

In [195]:
creative_rev = creative * creative_roas
audience_rev = audience * audience_roas
funnel_rev = funnel * funnel_roas
reserve_rev = reserve * reserve_roas

In [196]:
total_rev = creative_rev + audience_rev + funnel_rev + reserve_rev
total_profit = total_rev - TOTAL_BUDGET
total_roi = (total_profit / TOTAL_BUDGET) * 100

In [197]:
print(f"\n═══ 6-WEEK PROJECTION ═══")
print(f"Total Spend:   ₹{TOTAL_BUDGET:,.2f}")
print(f"Total Revenue: ₹{total_rev:,.2f}")
print(f"Total Profit:  ₹{total_profit:,.2f}")
print(f"Overall ROI:   {total_roi:.2f}%")


═══ 6-WEEK PROJECTION ═══
Total Spend:   ₹200,000.00
Total Revenue: ₹655,263.16
Total Profit:  ₹455,263.16
Overall ROI:   227.63%


In [198]:
## TEST & LEARN
#Test & Learn campaigns = 20% of whole budget
#campaigns budget = 200000

In [199]:
#6-MONTH PROJECTION (40% success rate)

In [200]:
Total_spend=200000
Expected_revenue= 264000
Expected_profit=64000
Overallroas=1.32
Overallroi=32

print(f"Total Spend : ₹{Total_spend}")
print(f"Expected Revenue : ₹{Expected_revenue}")
print(f"Expected Profit : ₹{Expected_profit}")
print(f"Overall ROAS : {Overallroas}x")
print(f"Overall ROI : {Overallroi:,.2f}%")

Total Spend : ₹200000
Expected Revenue : ₹264000
Expected Profit : ₹64000
Overall ROAS : 1.32x
Overall ROI : 32.00%


In [201]:
## BUFFER
#BUFFER campaigns = 10% of whole budget
#BUFFER budget=200000
#Festival Season
#Winning Test
#Competitor Attack
#Performance Dip

In [202]:
#6-MONTH PROJECTION (50% success rate)

In [203]:
BUFFER_Total_spend=100000
BUFFER_Expected_revenue= 205000
BUFFER_Expected_profit=105000
BUFFER_Overall_roas=2.05
BUFFER_Overall_roi=105

print(f"Total Spend : ₹{BUFFER_Total_spend}")
print(f"Expected Revenue : ₹{BUFFER_Expected_revenue}")
print(f"Expected Profit : ₹{BUFFER_Expected_profit}")
print(f"Overall ROAS : {BUFFER_Overall_roas}x")
print(f"Overall ROI : {BUFFER_Overall_roi:,.2f}%")

Total Spend : ₹100000
Expected Revenue : ₹205000
Expected Profit : ₹105000
Overall ROAS : 2.05x
Overall ROI : 105.00%


In [204]:
#final Summary

In [205]:
Total_budget=1000000
Total_revenue= estimate_revenue + total_rev + Expected_revenue + BUFFER_Expected_revenue
Total_profit= estimate_profit + total_profit + Expected_profit + BUFFER_Expected_profit
Overall_roas= Total_revenue/Total_budget
Overall_roi = (Total_profit/Total_budget)*100

print(f"Total Spend : ₹{Total_budget}")
print(f"Total Revenue : ₹{Total_revenue:,.2f}")
print(f"Toatal Profit : ₹{Total_profit:,.2f}")
print(f"Overall ROAS : {Overall_roas:,.2f}x")
print(f"Overall ROI : {Overall_roi:,.2f}%")

Total Spend : ₹1000000
Total Revenue : ₹3,024,172.89
Toatal Profit : ₹2,024,172.89
Overall ROAS : 3.02x
Overall ROI : 202.42%
